# BioBERT Fine-Tuning on MACCROBAT

Model: `dmis-lab/biobert-base-cased-v1.2`  
Dataset: `ktgiahieu/maccrobat2018_2020`

This notebook fine-tunes BioBERT for clinical named entity recognition using MACCROBAT's token-level BIO annotations. It follows the same workflow as `train_biobert_maccrobat.py`, but keeps each stage visible for inspection and experimentation.

## Workflow

1. Download and validate MACCROBAT.
2. Create deterministic train and test splits.
3. Build the 82-label BIO mapping.
4. Tokenize long documents into overlapping BioBERT windows.
5. Align word labels with the first subtoken of each word.
6. Fine-tune BioBERT on the training split.
7. Evaluate once on the test split and run sample inference.

## Imports and Configuration

Select the `dl-lab` Python 3.11 kernel before running the notebook. The helper functions are imported from the CLI script so that the notebook and terminal workflow use identical preprocessing and metrics.

In [3]:
from __future__ import annotations

import math
import sys
from pathlib import Path

import pandas as pd
import torch
from datasets import load_dataset
from transformers import (
    AutoModelForTokenClassification,
    AutoTokenizer,
    DataCollatorForTokenClassification,
    Trainer,
    TrainingArguments,
    pipeline,
    set_seed,
)

PROJECT_ROOT = Path.cwd().resolve()
while not (PROJECT_ROOT / "models/biobert_maccrobat/train_biobert_maccrobat.py").exists():
    if PROJECT_ROOT == PROJECT_ROOT.parent:
        raise FileNotFoundError("Run this notebook from inside the Deep-learning-Lab project.")
    PROJECT_ROOT = PROJECT_ROOT.parent

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from models.biobert_maccrobat.train_biobert_maccrobat import (
    DEFAULT_DATASET,
    DEFAULT_MODEL,
    build_compute_metrics,
    collect_labels,
    make_splits,
    tokenize_and_align,
    validate_dataset,
)

CONFIG = {
    "model_name": DEFAULT_MODEL,
    "dataset_name": DEFAULT_DATASET,
    "output_dir": PROJECT_ROOT / "outputs/biobert-maccrobat",
    "epochs": 50.0,
    "learning_rate": 2e-5,
    "train_batch_size": 8,
    "eval_batch_size": 8,
    "gradient_accumulation_steps": 1,
    "weight_decay": 0.01,
    "warmup_ratio": 0.1,
    "max_length": 512,
    "stride": 64,
    "seed": 42,
}

set_seed(CONFIG["seed"])
CONFIG

{'model_name': 'dmis-lab/biobert-base-cased-v1.2',
 'dataset_name': 'ktgiahieu/maccrobat2018_2020',
 'output_dir': PosixPath('/Users/kabilan/Documents/DL Lab/Deep-learning-Lab/outputs/biobert-maccrobat'),
 'epochs': 50.0,
 'learning_rate': 2e-05,
 'train_batch_size': 8,
 'eval_batch_size': 8,
 'gradient_accumulation_steps': 1,
 'weight_decay': 0.01,
 'warmup_ratio': 0.1,
 'max_length': 512,
 'stride': 64,
 'seed': 42}

## Load and Inspect MACCROBAT

MACCROBAT contains clinical documents represented as parallel `tokens` and `tags` sequences. Every token must have exactly one BIO label.

In [4]:
raw_dataset = load_dataset(CONFIG["dataset_name"], split="train")
validate_dataset(raw_dataset)

print(raw_dataset)
pd.DataFrame(
    {
        "token": raw_dataset[0]["tokens"][:30],
        "tag": raw_dataset[0]["tags"][:30],
    }
)

Dataset({
    features: ['tokens', 'tags'],
    num_rows: 400
})


,token,tag
0,A,O
1,68,B-Age
2,-,I-Age
3,year,I-Age
4,-,I-Age
5,old,I-Age
6,female,B-Sex
7,nonsmoker,B-History
8,",",O
9,nondrinker,B-History


## Train, Validation, and Test Splits

The split happens at document level before tokenization, preventing windows from the same clinical document from leaking across splits.

In [5]:
splits = make_splits(
    raw_dataset,
    validation_size=0.0,
    test_size=0.2,
    seed=CONFIG["seed"],
)

pd.DataFrame(
    [{"split": name, "documents": len(dataset)} for name, dataset in splits.items()]
)

,split,documents
0,train,320
1,test,80


## BIO Label Mapping

BioBERT predicts numeric class IDs. These mappings translate between those IDs and labels such as `B-Disease_disorder`, `I-Disease_disorder`, and `O`.

In [6]:
label_names = collect_labels(raw_dataset)
label_to_id = {label: index for index, label in enumerate(label_names)}
id_to_label = {index: label for label, index in label_to_id.items()}

print(f"Number of labels: {len(label_names)}")
pd.DataFrame(label_to_id.items(), columns=["label", "id"] )

Number of labels: 82


,label,id
0,O,0
1,B-Activity,1
2,B-Administration,2
3,B-Age,3
4,B-Area,4
...,...,...
77,I-Texture,77
78,I-Therapeutic_procedure,78
79,I-Time,79
80,I-Volume,80


## Tokenization and Label Alignment

Documents longer than 512 model tokens are divided into overlapping windows. Only the first subtoken of each original word receives its BIO label; special tokens and later subtokens receive `-100` so the loss function ignores them.

In [7]:
tokenizer = AutoTokenizer.from_pretrained(CONFIG["model_name"], use_fast=True)
if not tokenizer.is_fast:
    raise ValueError("A fast tokenizer is required for label alignment.")

tokenized_splits = splits.map(
    tokenize_and_align,
    batched=True,
    remove_columns=splits["train"].column_names,
    fn_kwargs={
        "tokenizer": tokenizer,
        "label_to_id": label_to_id,
        "max_length": CONFIG["max_length"],
        "stride": CONFIG["stride"],
    },
    desc="Tokenizing and aligning BIO labels",
)

pd.DataFrame(
    [
        {"split": name, "token_windows": len(dataset)}
        for name, dataset in tokenized_splits.items()
    ]
)

,split,token_windows
0,train,620
1,test,160


In [8]:
example = tokenized_splits["train"][0]
example_tokens = tokenizer.convert_ids_to_tokens(example["input_ids"])
example_labels = [
    id_to_label[label_id] if label_id != -100 else "IGNORED"
    for label_id in example["labels"]
]
pd.DataFrame({"model_token": example_tokens, "training_label": example_labels}).head(40)

,model_token,training_label
0,[CLS],IGNORED
1,a,O
2,13,B-Age
3,year,I-Age
4,-,I-Age
5,old,I-Age
6,boy,B-Sex
7,presented,B-Clinical_event
8,with,O
9,a,O


## Create BioBERT for Token Classification

The pretrained BioBERT encoder is reused and a new 82-class token-classification layer is initialized. A warning about newly initialized classifier weights is expected because MACCROBAT fine-tuning teaches this new layer.

In [9]:
model = AutoModelForTokenClassification.from_pretrained(
    CONFIG["model_name"],
    num_labels=len(label_names),
    id2label=id_to_label,
    label2id=label_to_id,
)

print(f"Loaded {CONFIG['model_name']} with {model.config.num_labels} output labels.")

[transformers] You passed `num_labels=82` which is incompatible to the `id2label` map of length `2`.


Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

[transformers] BertForTokenClassification LOAD REPORT from: dmis-lab/biobert-base-cased-v1.2
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
bert.pooler.dense.weight                   | UNEXPECTED | 
cls.predictions.decoder.bias               | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
bert.pooler.dense.bias                     | UNEXPECTED | 
cls.predictions.decoder.weight             | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED:	

Loaded dmis-lab/biobert-base-cased-v1.2 with 82 output labels.


## Configure the Trainer

With an 80/20 train-test split and no validation holdout, training runs for the configured epochs and the saved model is evaluated once on the test split afterward.

In [10]:
steps_per_epoch = math.ceil(
    len(tokenized_splits["train"])
    / (CONFIG["train_batch_size"] * CONFIG["gradient_accumulation_steps"])
)
warmup_steps = round(steps_per_epoch * CONFIG["epochs"] * CONFIG["warmup_ratio"])

training_args = TrainingArguments(
    output_dir=str(CONFIG["output_dir"]),
    num_train_epochs=CONFIG["epochs"],
    learning_rate=CONFIG["learning_rate"],
    per_device_train_batch_size=CONFIG["train_batch_size"],
    per_device_eval_batch_size=CONFIG["eval_batch_size"],
    gradient_accumulation_steps=CONFIG["gradient_accumulation_steps"],
    weight_decay=CONFIG["weight_decay"],
    warmup_steps=warmup_steps,
    eval_strategy="no",
    save_strategy="epoch",
    logging_strategy="steps",
    logging_steps=10,
    load_best_model_at_end=False,
    metric_for_best_model="f1",
    greater_is_better=True,
    save_total_limit=2,
    report_to="none",
    seed=CONFIG["seed"],
    data_seed=CONFIG["seed"],
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_splits["train"],
    eval_dataset=None,
    processing_class=tokenizer,
    data_collator=DataCollatorForTokenClassification(tokenizer),
    compute_metrics=build_compute_metrics(id_to_label),
)

print(f"Using device: {trainer.args.device}")
print(f"Training windows: {len(tokenized_splits['train'])}")
print(f"Steps per epoch: {steps_per_epoch}")
print(f"Warmup steps: {warmup_steps}")

Using device: mps
Training windows: 620
Steps per epoch: 78
Warmup steps: 390


## Train BioBERT

This is the computationally expensive cell. On Apple Silicon it can use MPS automatically; on a CUDA machine, add `fp16=True` to `TrainingArguments` if the GPU supports it.

In [11]:
train_result = trainer.train()
trainer.save_model()
tokenizer.save_pretrained(CONFIG["output_dir"])
trainer.log_metrics("train", train_result.metrics)
trainer.save_metrics("train", train_result.metrics)
train_result.metrics

/opt/anaconda3/envs/dl-lab/lib/python3.11/site-packages/torch/utils/data/dataloader.py:752: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)


Step,Training Loss
10,4.350997
20,4.261953
30,4.108138
40,3.864499
50,3.489719
60,2.832454
70,1.950586
80,1.488283
90,1.542233
100,1.563885


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

/opt/anaconda3/envs/dl-lab/lib/python3.11/site-packages/torch/utils/data/dataloader.py:752: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

...

/opt/anaconda3/envs/dl-lab/lib/python3.11/site-packages/torch/utils/data/dataloader.py:752: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

***** train metrics *****
  epoch                    =        50.0
  total_flos               =   7547127GF
  train_loss               =      0.1988
  train_runtime            = 21:31:49.43
  train_samples_per_second =         0.4
  train_steps_per_second   =        0.05


{'train_runtime': 77509.4329,
 'train_samples_per_second': 0.4,
 'train_steps_per_second': 0.05,
 'total_flos': 8103665945449968.0,
 'train_loss': 0.1988264964024226,
 'epoch': 50.0}

## Final Test Evaluation

Run this once after model selection. The test split was not used to choose checkpoints or tune hyperparameters.

In [14]:
test_metrics = trainer.evaluate(
    tokenized_splits["test"],
    metric_key_prefix="test",
)
trainer.log_metrics("test", test_metrics)
trainer.save_metrics("test", test_metrics)
pd.DataFrame([test_metrics]).T.rename(columns={0: "value"})

/opt/anaconda3/envs/dl-lab/lib/python3.11/site-packages/torch/utils/data/dataloader.py:752: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)


Training Loss,Validation Loss,Step,Precision,Recall,F1,Accuracy
0.006672,0.372981,3900,0.841129,0.935296,0.885717,0.955377


***** test metrics *****
  test_accuracy  = 0.9554
  test_f1        = 0.8857
  test_loss      =  0.373
  test_precision = 0.8411
  test_recall    = 0.9353


,value
test_loss,0.372981
test_precision,0.841129
test_recall,0.935296
test_f1,0.885717
test_accuracy,0.955377


## Sample Clinical NER

The pipeline groups BIO token predictions into readable entity spans. Change `input_text` to test another sentence.

In [13]:
input_text = (
    "A 67-year-old woman with hypertension presented with severe chest pain. "
    "She was given aspirin 325 mg and underwent coronary angiography."
)

ner_pipeline = pipeline(
    task="token-classification",
    model=trainer.model,
    tokenizer=tokenizer,
    aggregation_strategy="simple",
)
predictions = ner_pipeline(input_text)

pd.DataFrame(
    [
        {
            "entity": item["entity_group"],
            "text": input_text[item["start"] : item["end"]],
            "start": item["start"],
            "end": item["end"],
            "score": round(float(item["score"]), 4),
        }
        for item in predictions
    ]
)

,entity,text,start,end,score
0,Age,67-year-old,2,13,0.9993
1,Sex,woman,14,19,0.9977
2,Sign_symptom,hypertension,25,37,0.3890
3,Severity,severe,53,59,0.6959
4,Sign_symptom,pain,66,70,0.9990
5,Medication,aspirin,86,93,0.8778
6,Dosage,325 mg,94,100,0.9959
7,Diagnostic_procedure,coronary angiography,115,135,0.9933
